In [ ]:
# @title ### **1. Setup & Installation**
!pip install biopython scikit-learn lime requests pandas numpy matplotlib seaborn -q
!pip install alphafold
!pip install colabfold
import pandas as pd
import numpy as np
from Bio.Blast import NCBIWWW
from Bio import SeqIO
import requests
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
import lime
import lime.lime_tabular
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from io import StringIO
import os

In [ ]:
import requests


def get_alphafold_structure(uniprot_id):
    # Direct URL without constructing filename dynamically
    url = f"https://alphafold.ebi.ac.uk/files/AF-{uniprot_id}-F1-model_v4.pdb"
    print(f"Trying to download: {url}")  # Debugging URL here

    try:
        response = requests.get(url)
        response.raise_for_status()  # Will raise an exception for 404 or other errors

        # Save the structure to a file
        with open(f"AF-{uniprot_id}-F1-model_v4.pdb", "wb") as f:
            f.write(response.content)
        print(f"AlphaFold structure for {uniprot_id} saved as AF-{uniprot_id}-F1-model_v4.pdb")
        return f"AF-{uniprot_id}-F1-model_v4.pdb"

    except requests.exceptions.RequestException as e:
        print(f"Failed: {url} ({e})")
        return None


def fetch_structure(input_data):
    print(f"Fetching structure for input: {input_data}")  # Debugging input
    # Check if the input is a UniProt ID
    if input_data.startswith("P") or input_data.startswith("Q"):  # If it looks like a UniProt ID
        structure = get_alphafold_structure(input_data)
        if structure:
            return structure

    else:
        print("Invalid input. Please provide a valid UniProt ID.")
        return None


# Test with a UniProt ID
if __name__ == "__main__":
    uniprot_id = "Q9I579"  # Example UniProt ID
    fetch_structure(uniprot_id)

In [ ]:
import requests
import time
import xml.etree.ElementTree as ET

def run_blastp(sequence):
    """
    Submits a protein sequence to the NCBI BLAST API and retrieves results.
    Filters results to only include hits with E-value < 0.001.
    """
    url = "https://blast.ncbi.nlm.nih.gov/Blast.cgi"

    # Submit BLAST job
    params = {
        "CMD": "Put",
        "DATABASE": "nr",
        "PROGRAM": "blastp",
        "QUERY": sequence,
        "FORMAT_TYPE": "XML"
    }
    response = requests.post(url, data=params)

    # Extract BLAST Request ID (RID)
    if "RID" not in response.text:
        raise Exception("Failed to submit BLAST job.")
    rid = response.text.split("RID = ")[1].split("\n")[0].strip()

    # Wait for BLAST to process
    print("Running BLAST search... Please wait.")


    # Poll BLAST for results until they're ready
    for attempt in range(60):  # Up to ~5 minutes
        print(f"Checking BLAST status (attempt {attempt + 1})...")
        status_check = requests.get(url, params={
            "CMD": "Get",
            "RID": rid,
            "FORMAT_OBJECT": "SearchInfo"
        })

        if "Status=READY" in status_check.text:
            print("BLAST results are ready!")
            break
        elif "Status=FAILED" in status_check.text:
            raise Exception("BLAST search failed.")
        elif "Status=UNKNOWN" in status_check.text:
            raise Exception("BLAST RID expired or is unknown.")

        time.sleep(10)  # Wait before trying again
    else:
        raise TimeoutError("BLAST results not ready after several attempts.")

    # Retrieve results
    result_params = {
        "CMD": "Get",
        "FORMAT_TYPE": "XML",
        "RID": rid
    }
    result_response = requests.get(url, params=result_params)

    # Print out the raw XML response for debugging
    print(result_response.text)

    return result_response.text  # Return raw XML output


import xml.etree.ElementTree as ET
import requests
import re

def convert_to_uniprot_id(accession):
    """Convert RefSeq or GenBank accession to a UniProt ID using UniProt API."""
    url = f"https://rest.uniprot.org/idmapping/stream?from=RefSeq_Protein,GenBank&to=UniProtKB&format=tsv&query={accession}"
    response = requests.get(url)

    if response.status_code == 200:
        lines = response.text.strip().split("\n")
        for line in lines[1:]:  # Skip the header
            parts = line.split("\t")
            if len(parts) > 1:
                return parts[1]  # Extract the UniProt ID
    return None  # No UniProt ID found

def extract_uniprot_id(hit_def):
    """Attempt to extract a UniProt ID from the Hit_def field."""
    match = re.search(r"([OPQ][0-9][A-Z0-9]{3}[0-9])", hit_def)  # Matches UniProt ID format
    if match:
        return match.group(1)
    return None


def parse_blast_results(blast_xml):
    """Parse BLAST XML results and extract a list of UniProt IDs with E-values."""
    try:
        root = ET.fromstring(blast_xml)
    except ET.ParseError as e:
        print(f"Error parsing XML: {e}")
        return [], None

    hits = root.findall(".//Hit")

    if not hits:
        print("No significant hits found.")
        return [], None

    protein_list = []
    first_uniprot_id = None

    for hit in hits:
        hit_def = hit.find("Hit_def").text  # Protein description
        hit_accession = hit.find("Hit_accession").text  # Alternative ID
        e_value = hit.find(".//Hsp_evalue")  # Get E-value
        e_value = float(e_value.text) if e_value is not None else None

        print(f"Raw Hit_def: {hit_def}")
        print(f"Raw Hit_accession: {hit_accession}")
        print(f"E-value: {e_value}")

        uniprot_id = extract_uniprot_id(hit_def)  # Try to extract from Hit_def

        if not uniprot_id and hit_accession:
            uniprot_id = convert_to_uniprot_id(hit_accession)  # Convert ID if needed

        if not uniprot_id:
            continue  # Skip if still no UniProt ID

        # Store first valid UniProt ID
        if not first_uniprot_id:
            first_uniprot_id = uniprot_id

        # Store valid results in the list
        if e_value is not None and e_value < 0.001:
            protein_list.append((uniprot_id, e_value))

    if not first_uniprot_id:
        print("No valid UniProt IDs found.")

    return protein_list, first_uniprot_id

In [ ]:
from Bio import SeqIO
import os

def is_valid_protein(sequence):

    valid_amino_acids = "ACDEFGHIKLMNPQRSTVWY"
    sequence = sequence.upper().replace(" ", "").replace("\n", "")
    return all(residue in valid_amino_acids for residue in sequence), sequence

def load_fasta_sequence(file_path):

    if not os.path.exists(file_path):
        print("Error: File not found.")
        return None
    with open(file_path, "r") as fasta_file:
        record = next(SeqIO.parse(fasta_file, "fasta"))
        return str(record.seq)

def get_user_sequence():

    choice = input("Do you want to enter a sequence manually (M) or provide a FASTA file (F)? ").strip().upper()

    if choice == "M":
        user_input = input("Enter the amino acid sequence: ").strip()
        valid, processed_sequence = is_valid_protein(user_input)
    elif choice == "F":
        file_path = input("Enter the FASTA file path: ").strip()
        fasta_sequence = load_fasta_sequence(file_path)
        if fasta_sequence:
            valid, processed_sequence = is_valid_protein(fasta_sequence)
        else:
            valid = False
    else:
        print("Invalid choice! Please enter 'M' for manual input or 'F' for FASTA file.")
        valid = False

    if valid:
        return processed_sequence
    else:
        print("Invalid protein sequence!")
        return None

In [ ]:
from Bio import SeqIO
import os

def is_valid_protein(sequence):

    valid_amino_acids = "ACDEFGHIKLMNPQRSTVWY"
    sequence = sequence.upper().replace(" ", "").replace("\n", "")
    return all(residue in valid_amino_acids for residue in sequence), sequence

def load_fasta_sequence(file_path):

    if not os.path.exists(file_path):
        print("Error: File not found.")
        return None
    with open(file_path, "r") as fasta_file:
        record = next(SeqIO.parse(fasta_file, "fasta"))
        return str(record.seq)

def get_user_sequence():

    choice = input("Do you want to enter a sequence manually (M) or provide a FASTA file (F)? ").strip().upper()

    if choice == "M":
        user_input = input("Enter the amino acid sequence: ").strip()
        valid, processed_sequence = is_valid_protein(user_input)
    elif choice == "F":
        file_path = input("Enter the FASTA file path: ").strip()
        fasta_sequence = load_fasta_sequence(file_path)
        if fasta_sequence:
            valid, processed_sequence = is_valid_protein(fasta_sequence)
        else:
            valid = False
    else:
        print("Invalid choice! Please enter 'M' for manual input or 'F' for FASTA file.")
        valid = False

    if valid:
        return processed_sequence
    else:
        print("Invalid protein sequence!")
        return None

In [ ]:
def parse_foldseek_m8(file_path, evalue_threshold=0.001, max_results=10):
    results = []

    with open(file_path, 'r') as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue

            parts = line.strip().split('\t')
            if len(parts) < 12:
                continue

            try:
                query_id = parts[0]
                target_id = parts[1]
                evalue = float(parts[11].replace(",", "").strip())
                bitscore = float(parts[11])

                if evalue < evalue_threshold:
                    results.append({
                        'query': query_id,
                        'target': target_id,
                        'evalue': bitscore,
                        'bitscore': bitscore
                    })
            except ValueError:
                continue

    results.sort(key=lambda x: (x['evalue'], -x['bitscore']))

    return results


def display_results(hits, top_n=20):
    if not hits:
        print("No hits found below the E-value threshold.")
        return

    print(f"\nTop {min(top_n, len(hits))} Foldseek Results:\n")
    for i, hit in enumerate(hits[:top_n], 1):
        print(f"[{i}] {hit['query']} → {hit['target']}")
        print(f"     E-value: {hit['evalue']}")


if __name__ == "__main__":
    main()

In [ ]:
import requests
import os


def search_pdb_by_sequence(sequence):
    """
    Search the RCSB Protein Data Bank (PDB) for structures matching the given protein sequence.

    Args:
        sequence (str): The amino acid sequence of the protein.

    Returns:
        list: A list of matching PDB IDs.
    """
    url = "https://search.rcsb.org/rcsbsearch/v1/query"
    query = {
        "query": {
            "type": "terminal",
            "service": "sequence",
            "parameters": {
                "evalue_cutoff": 0.01,
                "identity_cutoff": 90,
                "target": "pdb_protein_sequence",
                "value": sequence
            }
        },
        "return_type": "entry"
    }

    response = requests.post(url, json=query)
    if response.status_code == 200:
        data = response.json()
        pdb_ids = [entry["identifier"] for entry in data.get("result_set", [])]
        return pdb_ids
    else:
        print(f"Failed to search PDB. Status Code: {response.status_code}")
        return []


def fetch_pdb_structure(pdb_id, output_directory="pdb_structures"):
    """
    Fetch PDB structure by PDB ID and save it as a .pdb file.

    Args:
        pdb_id (str): The PDB ID of the protein structure.
        output_directory (str): Directory where the structure will be saved.

    Returns:
        None
    """
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    # Ensure the output directory exists
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    output_file = os.path.join(output_directory, f"{pdb_id}.pdb")
    try:
        print(f"Fetching PDB structure for ID: {pdb_id}...")
        response = requests.get(url, stream=True)
        if response.status_code == 200:
            with open(output_file, 'wb') as file:
                for chunk in response.iter_content(chunk_size=8192):
                    file.write(chunk)
            print(f"AlphaFold structure saved as {output_file}")
        else:
            print(f"Failed to fetch PDB structure for ID {pdb_id}. Status Code: {response.status_code}")
    except requests.RequestException as e:
        print(f"An error occurred while fetching the PDB structure: {e}")

In [ ]:
import requests
import pandas as pd
from io import StringIO
from time import sleep

def fetch_uniprot_data(query="reviewed:true", limit=100, max_retries=3):
    """
    Fetches protein data from UniProt's current REST API

    Args:
        query (str): Search query (default: reviewed proteins)
        limit (int): Maximum number of results
        max_retries (int): Number of retry attempts

    Returns:
        pd.DataFrame with columns: uniprot_id, sequence, function
    """
    base_url = "https://rest.uniprot.org/uniprotkb/search"

    params = {
        'query': query,
        'fields': 'accession,sequence,protein_name',
        'size': limit,
        'format': 'tsv'
    }

    headers = {
        'Accept': 'text/plain',
        'User-Agent': 'Python-Script'
    }

    for attempt in range(max_retries):
        try:
            response = requests.get(base_url, params=params, headers=headers)
            response.raise_for_status()

            # Parse TSV data
            df = pd.read_csv(StringIO(response.text), sep='\t')
            df.columns = ['uniprot_id', 'sequence', 'function']

            # Validate data
            if df.empty:
                raise ValueError("Empty response - check your query")
            if pd.isna(df['sequence']).any():
                raise ValueError("Missing sequence data")

            return df

        except Exception as e:
            print(f"Attempt {attempt+1} failed: {str(e)}")
            if attempt == max_retries - 1:
                raise
            sleep(5)  # Wait before retry

    return pd.DataFrame(columns=['uniprot_id', 'sequence', 'function'])

# Usage example
try:
    print("Fetching protein data...")
    protein_df = fetch_uniprot_data(limit=100)
    print(f"Success! Downloaded {len(protein_df)} proteins")
    print(protein_df.head())

except Exception as e:
    print(f"Failed to fetch data: {str(e)}")

In [ ]:
!apt install diamond-aligner -qq
!wget https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/uniprot_sprot.fasta.gz
!gunzip uniprot_sprot.fasta.gz
!diamond makedb --in uniprot_sprot.fasta -d swissprot

In [ ]:
# @title ### **1. INSTALL REQUIRED TOOLS**
!apt update && apt install -y diamond-aligner
!wget https://mmseqs.com/foldseek/foldseek-linux-avx2.tar.gz
!tar xvfz foldseek-linux-avx2.tar.gz
!mv foldseek/bin/foldseek /usr/local/bin/
!rm -rf foldseek*a

# Install AlphaFold dependencies (if not already present)
!pip install --upgrade pip
!pip install alphafold-colabfold

# @title ### **2. SETUP DATABASES**
import os
from pathlib import Path

# Create working directories
Path("databases").mkdir(exist_ok=True)
Path("tmp").mkdir(exist_ok=True)

# Download SwissProt (for Diamond BLAST)
if not os.path.exists("uniprot_sprot.fasta"):
    print("Downloading SwissProt...")
    !wget https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/uniprot_sprot.fasta.gz
    !gunzip uniprot_sprot.fasta.gz
    !diamond makedb --in uniprot_sprot.fasta -d databases/swissprot --threads 4

# Download Foldseek example database
if not os.path.exists("databases/example"):
    print("Downloading Foldseek example database...")
    !wget -O example_db.tar.gz https://mmseqs.com/foldseek/example_db.tar.gz
    !tar xvfz example_db.tar.gz -C databases/
    !rm example_db.tar.gz

# @title ### **3. FEATURE EXTRACTION PIPELINE**
import pandas as pd
import numpy as np
import subprocess
from concurrent.futures import ThreadPoolExecutor
from colabfold import alphafold

class ProteinFeatureExtractor:
    def __init__(self):
        self.validate_databases()

    def validate_databases(self):
        """Verify all required databases exist"""
        required_files = {
            'Diamond': 'databases/swissprot.dmnd',
            'Foldseek': 'databases/example'
        }
        for name, path in required_files.items():
            if not os.path.exists(path):
                raise FileNotFoundError(f"{name} database not found at {path}")

    def run_command(self, cmd):
        """Execute command with error handling"""
        try:
            result = subprocess.run(
                cmd, shell=True, check=True,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True
            )
            return True, result.stdout
        except subprocess.CalledProcessError as e:
            print(f"Command failed: {e.cmd}")
            print(f"Error: {e.stderr}")
            return False, None

    def extract_blast_features(self, sequence, tmp_prefix):
        """Extract sequence similarity features using Diamond BLAST"""
        query_file = f"{tmp_prefix}.fa"
        blast_out = f"{tmp_prefix}_blast.out"

        features = {'blast_evalue': 0, 'blast_identity': 0}

        with open(query_file, 'w') as f:
            f.write(f">query\n{sequence}\n")

        cmd = (
            f"diamond blastp -q {query_file} -d databases/swissprot "
            f"--outfmt 6 qseqid sseqid pident evalue --threads 2 "
            f"--max-target-seqs 1 > {blast_out}"
        )
        success, _ = self.run_command(cmd)

        if success and os.path.getsize(blast_out) > 0:
            try:
                blast_results = pd.read_csv(blast_out, sep='\t', header=None)
                if not blast_results.empty:
                    features['blast_identity'] = float(blast_results.iloc[0, 2])
                    evalue = float(blast_results.iloc[0, 3])
                    features['blast_evalue'] = -np.log10(evalue) if evalue > 0 else 0
            except Exception as e:
                print(f"BLAST results processing error: {e}")

        for f in [query_file, blast_out]:
            if os.path.exists(f):
                os.remove(f)
        return features

    def extract_foldseek_features(self, sequence, tmp_prefix):
        """Extract structural similarity features using Foldseek"""
        query_file = f"{tmp_prefix}.fa"
        fs_out = f"{tmp_prefix}_fs.out"

        features = {'fs_tmscore': 0}

        with open(query_file, 'w') as f:
            f.write(f">query\n{sequence}\n")

        cmd = (
            f"foldseek easy-search {query_file} databases/example "
            f"{fs_out} tmp --threads 2 --max-accept 1"
        )
        success, _ = self.run_command(cmd)

        if success and os.path.getsize(fs_out) > 0:
            try:
                fs_results = pd.read_csv(fs_out, sep='\t', header=None)
                if not fs_results.empty and len(fs_results.columns) > 10:
                    features['fs_tmscore'] = float(fs_results.iloc[0, 10])
            except Exception as e:
                print(f"Foldseek results processing error: {e}")

        for f in [query_file, fs_out]:
            if os.path.exists(f):
                os.remove(f)
        return features

    def run_alphafold(self, sequence, tmp_prefix):
        """Predict structure using AlphaFold (via ColabFold)"""
        features = {'af_plddt': 0, 'af_ptm': 0}

        try:
            # Run AlphaFold prediction
            result = alphafold.predict_structure(
                sequence,
                f"{tmp_prefix}_af",  # Output prefix
                model_type="auto"
            )

            # Extract features (pLDDT and predicted TM-score)
            features['af_plddt'] = result['plddt'].mean()
            features['af_ptm'] = result['ptm']

        except Exception as e:
            print(f"AlphaFold prediction error: {e}")

        return features

    def extract_features(self, sequence):
        """Extract all features (BLAST + Foldseek + AlphaFold)"""
        tmp_prefix = f"tmp/seq_{os.getpid()}_{hash(sequence)}"

        # Get features from all tools
        blast_features = self.extract_blast_features(sequence, tmp_prefix)
        fs_features = self.extract_foldseek_features(sequence, tmp_prefix)
        af_features = self.run_alphafold(sequence, tmp_prefix)

        # Combine all features
        combined_features = {
            **blast_features,
            **fs_features,
            **af_features
        }

        return combined_features